In [1]:
# HYPER PARAM
PROJECT_NAME = "RobocupTrajectoryPrediction_0902_1642"  # wandbのプロジェクト名
GROUP_NAME = "BiGRU"  # wandbのグループ名
MODEL_NAME = "BiGRU_1"  # 保存時のファイル名
BATCH_SIZE = 512
EPOCHS = 200
HIDDEN_DIM = 256
NUM_LAYER = 4
LEARNING_RATE = 0.001

In [2]:
import joblib
import os

import visualizer

from tqdm.notebook import tqdm
import datasets
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
from torch import nn
import torchmetrics
import pytorch_lightning as pl

In [3]:
torch.set_float32_matmul_precision("high")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [4]:
class MinMax:
    def __init__(self, min, max):
        self.min = min
        self.max = max
        if self.min >= self.max:
            raise ValueError("min must be less than max")

    def __call__(self, x):
        return (x - self.min) / (self.max - self.min)

    def inverse(self, x):
        return x * (self.max - self.min) + self.min

    def __repr__(self):
        return f"MinMax({self.min}, {self.max})"

In [5]:
train = np.load("/mnt/data1/yano/work/SoccerCompetition2025/datas_20260828_015504/train.npy")
cols = np.load("/mnt/data1/yano/work/SoccerCompetition2025/datas_20260828_015504/train_col.npy", allow_pickle=True)
min_max_d = np.load("/mnt/data1/yano/work/SoccerCompetition2025/datas_20260828_015504/min_max_d.npy", allow_pickle=True).item()
test = np.load("/mnt/data1/yano/work/SoccerCompetition2025/datas_20260828_015504/test.npy")

In [6]:
# 2026_0904 追加

# class NpyDataset:
#     """
#     分割保存された.npyファイルを、
#     1つのDatasetのように扱うためのクラス。
#     """

#     def __init__(self, files):
#         self.files = files

#         self._shapes = []
#         self._lengths = []

#         for file in self.files:
#             arr = np.load(file, mmap_mode="r")
#             self._shapes.append(arr.shape)
#             self._lengths.append(arr.shape[0])

#         self._length = sum(self._lengths)

#     def __len__(self):
#         return self._length

#     def __getitem__(self, index):

#         if index < 0:
#             index += len(self)

#         if index < 0 or index >= len(self):
#             raise IndexError("index out of range")

#         current = 0

#         for file, length in zip(
#             self.files,
#             self._lengths
#         ):

#             if index < current + length:

#                 local_index = index - current

#                 arr = np.load(
#                     file,
#                     mmap_mode="r"
#                 )

#                 return arr[local_index]

#             current += length

#         raise IndexError("index out of range")

#     def iter_chunks(self):

#         for file in self.files:

#             yield np.load(
#                 file,
#                 mmap_mode="r"
#             )

In [7]:
# 2026_0904 追加

# from pathlib import Path
# import numpy as np


# data_dir = Path(
#     "/mnt/data1/yano/work/SoccerCompetition2025/"
#     "datas_20260828_015504"
# )


# # 分割ファイル
# train_files = sorted(
#     (data_dir / "train").glob("*.npy")
# )

# test_files = sorted(
#     (data_dir / "test").glob("*.npy")
# )


# # Dataset
# train = NpyDataset(train_files)
# test = NpyDataset(test_files)


# # columns
# train_col = np.load(
#     data_dir / "train_col.npy",
#     allow_pickle=True
# )

# test_col = np.load(
#     data_dir / "test_col.npy",
#     allow_pickle=True
# )


# # MinMax
# min_max_d = np.load(
#     data_dir / "min_max_d.npy",
#     allow_pickle=True
# ).item()


# # 既存コードが cols を要求する場合
# cols = train_col


# print("train:", len(train))
# print("test :", len(test))
# print("train_col:", train_col.shape)
# print("test_col :", test_col.shape)

In [10]:
# 2026_0904 追加
# print("train files:", len(train_files))
# print("test files :", len(test_files))

# print("train rows:", len(train))
# print("test rows :", len(test))

# print("first train shape:",
#       np.load(train_files[0], mmap_mode="r").shape)

# print("first train dtype:",
#       np.load(train_files[0], mmap_mode="r").dtype)

In [11]:
X_train, y_train = train[:, :20, :], train[:, 20:, :]
X_test, y_test = test[:, :20, :], test[:, 20:, :]

print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(179501, 20, 223) (179501, 30, 223) (44847, 20, 223) (44847, 30, 223)


In [12]:
class EuclideanDistance(torchmetrics.Metric):
    """
    終端誤差を計算する
    """

    def __init__(self, cols, min_max_d, **kwargs):
        super().__init__(**kwargs)
        self.cols = cols
        self.min_max_d = min_max_d
        self.add_state("sum", default=torch.tensor(0.0), dist_reduce_fx="sum")
        self.add_state("count", default=torch.tensor(0), dist_reduce_fx="sum")
        self._mins = torch.tensor(
            [min_max_d[col].min for col in cols], dtype=torch.float32
        )
        self._maxs = torch.tensor(
            [min_max_d[col].max for col in cols], dtype=torch.float32
        )
        self._scale = self._maxs - self._mins

    def update(self, preds, target):
        indices = torch.cat([torch.tensor([0]), torch.arange(5, 201, 9)])
        final_preds = preds[:, -1, :]
        final_target = target[:, -1, :]

        inversed_preds = torch.zeros_like(final_preds)
        inversed_target = torch.zeros_like(final_target)

        mins = self._mins.to(device=device, dtype=final_preds.dtype)
        scale = self._scale.to(device=device, dtype=final_preds.dtype)

        inversed_preds = final_preds.clone()
        inversed_target = final_target.clone()
        n_cols = len(self.cols)
        inversed_preds[:, :n_cols] = final_preds[:, :n_cols] * scale + mins
        inversed_target[:, :n_cols] = final_target[:, :n_cols] * scale + mins

        errors = torch.sqrt(
            (inversed_preds[:, indices] - inversed_target[:, indices]) ** 2
            + (inversed_preds[:, indices + 1] - inversed_target[:, indices + 1]) ** 2
        )
        self.sum += torch.sum(errors)
        # self.count += errors.size(0)
        self.count += errors.numel()

    def compute(self):
        return self.sum / self.count

In [13]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True, num_workers=16
)
val_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=16
)

In [14]:
# 2026_0904 追加
# arr = np.load(train_files[0], mmap_mode="r")

# print(arr.shape)
# print(arr.dtype)

NameError: name 'train_files' is not defined

In [17]:
# 2026_0904 追加
# import numpy as np
# import torch
# from torch.utils.data import Dataset, DataLoader


# class NpySequenceDataset(Dataset):
#     """
#     1つの.npyファイルを1サンプルとして扱うDataset。

#     各.npy:
#         (50, 223)

#     入力:
#         先頭20時刻 → (20, 223)

#     教師:
#         後半30時刻 → (30, 223)
#     """

#     def __init__(self, files):
#         self.files = list(files)

#     def __len__(self):
#         return len(self.files)

#     def __getitem__(self, index):

#         file = self.files[index]

#         # ファイル全体をRAMに読み込まない
#         arr = np.load(file, mmap_mode="r")

#         # 入力20時刻
#         x = np.asarray(
#             arr[:20],
#             dtype=np.float32
#         ).copy()

#         # 教師30時刻
#         y = np.asarray(
#             arr[20:],
#             dtype=np.float32
#         ).copy()

#         x = torch.from_numpy(x)
#         y = torch.from_numpy(y)

#         return x, y

In [18]:
# train_dataset = NpySequenceDataset(train_files)
# val_dataset = NpySequenceDataset(test_files)

# train_loader = DataLoader(
#     train_dataset,
#     batch_size=BATCH_SIZE,
#     shuffle=True,
#     pin_memory=True,
#     num_workers=16,
#     persistent_workers=True,
# )

# val_loader = DataLoader(
#     val_dataset,
#     batch_size=BATCH_SIZE,
#     shuffle=False,
#     pin_memory=True,
#     num_workers=16,
#     persistent_workers=True,
# )

In [19]:
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import WandbLogger


class LitBiGRU(pl.LightningModule):
    def __init__(
        self, input_dim, hidden_dim, output_dim, num_layers, seq_length, lr=0.001
    ):
        super().__init__()
        self.save_hyperparameters()
        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
        )
        self.fc = nn.Linear(hidden_dim * 2, output_dim * seq_length)
        self.criterion = nn.MSELoss()
        self.euclidean_distance = EuclideanDistance(cols, min_max_d)
        self.train_losses = []
        self.val_losses = []

    def forward(self, x):
        out, _ = self.gru(x)
        last_out = out[:, -1, :]
        output = self.fc(last_out)
        return output.view(-1, 30, self.hparams.output_dim)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        self.log("train_loss", loss, on_step=False, on_epoch=True)
        self.log(
            "train_euclidean_distance",
            self.euclidean_distance(y_hat, y),
            on_step=False,
            on_epoch=True,
        )
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        self.log("val_loss", loss, on_step=False, on_epoch=True)
        self.log(
            "val_euclidean_distance",
            self.euclidean_distance(y_hat, y),
            on_step=False,
            on_epoch=True,
        )
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.lr)
        return optimizer


checkpoint_callback = ModelCheckpoint(
    monitor="val_euclidean_distance",
    dirpath=f"checkpoints/{MODEL_NAME}",
    filename="bilstm-{epoch:02d}-{val_euclidean_distance:.4f}",
    save_top_k=3,
    mode="min",
)

wandb_logger = WandbLogger(
    project=PROJECT_NAME,
    log_model=True,
    save_code=True,
    save_dir="logs/",
    name=MODEL_NAME,
    group=GROUP_NAME,
)


trainer = pl.Trainer(
    max_epochs=EPOCHS,
    accelerator="gpu",
    devices=1,
    callbacks=[checkpoint_callback],
    # logger=wandb_logger,  # wandbを使わない場合はコメントアウトしてください
)

input_dim = X_train.shape[2]
hidden_dim = HIDDEN_DIM
output_dim = y_train.shape[2]
seq_length = y_train.shape[1]
num_layers = NUM_LAYER
learning_rate = LEARNING_RATE

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [20]:
import torch
import torch.nn as nn
import pytorch_lightning as pl

from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import WandbLogger


class LitBiGRU(pl.LightningModule):

    def __init__(
        self,
        input_dim,
        hidden_dim,
        output_dim,
        num_layers,
        seq_length,
        lr=0.001,
    ):
        super().__init__()

        self.save_hyperparameters()

        # BiGRU
        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
        )

        # 双方向なので hidden_dim * 2
        self.fc = nn.Linear(
            hidden_dim * 2,
            output_dim * seq_length,
        )

        # 損失関数
        self.criterion = nn.MSELoss()

        # 評価指標
        self.euclidean_distance = EuclideanDistance(
            cols,
            min_max_d,
        )

    def forward(self, x):

        # x:
        # (batch, 20, 223)

        out, _ = self.gru(x)

        # 最後の時刻の出力
        # (batch, hidden_dim * 2)
        last_out = out[:, -1, :]

        # (batch, output_dim * seq_length)
        output = self.fc(last_out)

        # (batch, 30, 223)
        output = output.view(
            -1,
            self.hparams.seq_length,
            self.hparams.output_dim,
        )

        return output

    def training_step(self, batch, batch_idx):

        x, y = batch

        # x:     (B, 20, 223)
        # y:     (B, 30, 223)
        # y_hat: (B, 30, 223)
        y_hat = self(x)

        # MSE
        loss = self.criterion(y_hat, y)

        self.log(
            "train_loss",
            loss,
            on_step=False,
            on_epoch=True,
            prog_bar=True,
        )

        # Euclidean Distance
        self.log(
            "train_euclidean_distance",
            self.euclidean_distance(y_hat, y),
            on_step=False,
            on_epoch=True,
        )

        return loss

    def validation_step(self, batch, batch_idx):

        x, y = batch

        y_hat = self(x)

        # MSE
        loss = self.criterion(y_hat, y)

        self.log(
            "val_loss",
            loss,
            on_step=False,
            on_epoch=True,
            prog_bar=True,
        )

        # Euclidean Distance
        self.log(
            "val_euclidean_distance",
            self.euclidean_distance(y_hat, y),
            on_step=False,
            on_epoch=True,
            prog_bar=True,
        )

        return loss

    def configure_optimizers(self):

        optimizer = torch.optim.Adam(
            self.parameters(),
            lr=self.hparams.lr,
        )

        return optimizer

In [21]:
input_dim = 223
hidden_dim = HIDDEN_DIM
output_dim = 223
seq_length = 30
num_layers = NUM_LAYER
learning_rate = LEARNING_RATE

In [22]:
model = LitBiGRU(
    input_dim, hidden_dim, output_dim, num_layers, seq_length, lr=learning_rate
)
trainer.fit(model, train_loader, val_loader)

2026-09-22 15:33:28.623976: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-22 15:33:28.709363: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1790058808.744849  231130 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1790058808.757234  231130 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-09-22 15:33:28.844171: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=200` reached.


In [23]:
# モデルの保存
import time

best_model_path = checkpoint_callback.best_model_path
model = LitBiGRU.load_from_checkpoint(best_model_path)
file_name = f"{MODEL_NAME}_{time.strftime('%Y%m%d%H%M%S')}.pth"

os.makedirs(f"models/{MODEL_NAME}", exist_ok=True)
torch.save(model.state_dict(), f"models/{MODEL_NAME}/{file_name}")